<a href="https://colab.research.google.com/github/ralazar98/food_model/blob/main/%D0%9F%D1%80%D0%B0%D0%BA%D1%82%D0%B8%D0%BA%D0%B0_%D0%95%D0%B4%D0%B0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import datasets
import pandas as pd
import ast
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import PassiveAggressiveClassifier,PassiveAggressiveRegressor
from sklearn.metrics import accuracy_score,mean_absolute_error, r2_score
from scipy.sparse import hstack ,csr_matrix
import numpy as np
import re
from xgboost import XGBRegressor

In [ ]:
# Загрузка датасета
dataset_food = datasets.load_dataset("Codatta/MM-Food-100K")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

MM-Food-100K.csv:   0%|          | 0.00/28.6M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/100000 [00:00<?, ? examples/s]

In [ ]:
# Формулы для преобразование столбцов

def ingredients_parse(ingredients):
    if not ingredients or pd.isna(ingredients):
        return[]
    try:
        remake_ingredient = ast.literal_eval(ingredients)
        if isinstance(remake_ingredient,list):
            return remake_ingredient
    except:
        pass

def portion_size_parse(portion_size):
    if not portion_size or pd.isna(portion_size):
        return {}
    try:
        remake_portion_size = ast.literal_eval(portion_size)

        if isinstance(remake_portion_size,dict):
            return remake_portion_size

        elif isinstance(remake_portion_size,list):
            dict_portion_size = {}
            for item in remake_portion_size:
                parts = str(item).split(':',1)
                ingredient = parts[0].strip()
                weight = parts[1].strip()
                dict_portion_size[ingredient] = weight
            return dict_portion_size
        else:
            return {}
    except:
        return {}

def nutritional_profile_parse(nutritional_profile):
  if not nutritional_profile or pd.isna(nutritional_profile):
      return {}
  try:
      remake_nutritional_profile = ast.literal_eval(nutritional_profile)

      if isinstance(remake_nutritional_profile,dict):
          return remake_nutritional_profile

      elif isinstance(remake_nutritional_profile,list):
          dict_portion_size = {}
          for item in remake_nutritional_profile:
              parts = str(item).split(':',1)
              ingredient = parts[0].strip()
              weight = parts[1].strip()
              dict_portion_size[ingredient] = weight
          return dict_portion_size
      else:
          return {}
  except:
      return {}

def extract_total_weight_from_portion(portion_size):
    if not isinstance(portion_size, dict):
        return np.nan

    if not portion_size:
        return np.nan

    total = 0.0

    for ingredient, weight_str in portion_size.items():
        weight_str = str(weight_str).lower().strip()

        # Ищем число в строке
        numbers = re.findall(r'(\d+(?:\.\d+)?)', weight_str)
        if not numbers:
            continue

        weight_value = float(numbers[0])

        # Определяем единицу измерения
        if 'kg' in weight_str:
            weight_value *= 1000
        elif 'g' in weight_str:
            pass
        elif 'ml' in weight_str:
            pass
        else:
            continue

        total += weight_value

    return total if total > 0 else np.nan

In [ ]:
# Преобразование ингредиентов

success_count = 0
fail_count = 0
empty_count = 0

for i in range(len(dataset_food['train'])):
    data_ingredients = dataset_food['train'][i].get('ingredients','')
    parsed_ingredients = ingredients_parse(data_ingredients)


In [ ]:
# Преобразование порций

success_count = 0
fail_count = 0
empty_count = 0

for i in range(len(dataset_food['train'])):
    data_portion_size = dataset_food['train'][i].get('portion_size','')
    parsed_portion_size = portion_size_parse(data_portion_size)


In [ ]:
# Преобразование нутреиновых показателей

success_count = 0
fail_count = 0
empty_count = 0

for i in range(len(dataset_food['train'])):
    data_nutritional_profile = dataset_food['train'][i].get('nutritional_profile','')
    parsed_nutritional_profile = nutritional_profile_parse(data_nutritional_profile)


In [ ]:
# Создаем откоректированнаый датасет

parsed_data =[]
for i in range(len(dataset_food['train'])):
    dataset_food_raw = dataset_food['train'][i]

    ingredients = ingredients_parse(dataset_food_raw.get('ingredients',''))
    portion_size = portion_size_parse(dataset_food_raw.get('portion_size',''))
    nutritional_profile = nutritional_profile_parse(dataset_food_raw.get('nutritional_profile',''))
    calories = nutritional_profile['calories_kcal']

    parsed_data.append({
        'dish_name': dataset_food_raw.get('dish_name', ''),
        'food_type': dataset_food_raw.get('food_type', ''),
        'ingredients': ingredients,
        'portion_size' : portion_size,
        'nutritional_profile' : nutritional_profile,
        'cooking_method' : dataset_food_raw.get('cooking_method', ''),
        'calories' : calories,
    })

df_food = pd.DataFrame(parsed_data)

✅ Создан DataFrame размером (100000, 7)


In [ ]:
df_food.head(5)

,dish_name,food_type,ingredients,portion_size,nutritional_profile,cooking_method,calories
0,Fried Chicken,Restaurant food,"[chicken, breading, oil]",{'chicken': '300g'},"{'fat_g': 25.0, 'protein_g': 30.0, 'calories_k...",Frying,400
1,Pho,Restaurant food,"[noodles, beef, basil, lime, green onions, chili]","{'noodles': '200g', 'beef': '100g', 'vegetable...","{'fat_g': 15.0, 'protein_g': 25.0, 'calories_k...",boiled,450
2,Pan-fried Dumplings,Restaurant food,"[dumplings, chili oil, soy sauce]","{'dumplings': '300g', 'sauce': '50g'}","{'fat_g': 15.0, 'protein_g': 20.0, 'calories_k...",Pan-frying,400
3,Bananas,Raw vegetables and fruits,[Bananas],{'Bananas': '10 pieces (about 1kg)'},"{'fat_g': 3.0, 'protein_g': 12.0, 'calories_kc...",Raw,1050
4,Noodle Stir-Fry,Restaurant food,"[noodles, chicken, vegetables, sauce]","{'noodles': '300g', 'chicken': '100g', 'vegeta...","{'fat_g': 20.0, 'protein_g': 25.0, 'calories_k...",stir-fried,600


In [ ]:
df_food_model = df_food.copy()

# Вес еды
df_food_model['total_weight'] = df_food_model['portion_size'].apply(extract_total_weight_from_portion)
df_food_model = df_food_model[df_food_model['total_weight'].notna()].copy()
X_total_weight = csr_matrix(df_food_model[['total_weight']].values)


# Ингредиенты
df_food_model['ingredient_text'] = df_food_model['ingredients'].apply(
    lambda x: ' '.join(x)
)

ingredients_tfidf_vector = TfidfVectorizer(
    max_features=2000,
    ngram_range=(1,2),
    min_df=2
)
X_ingredients_tfidf = ingredients_tfidf_vector.fit_transform(df_food_model['ingredient_text'])

# Метод готовки
df_food_model['cooking_method_clean'] = df_food_model['cooking_method'].fillna('').astype(str)
cooking_method_tfidf_vector = TfidfVectorizer(
    max_features=50,
    ngram_range=(1,1),
    min_df=1
)
X_cooking_method_tfidf = cooking_method_tfidf_vector.fit_transform(df_food_model['cooking_method_clean'])

# Тип еды
df_food_model['food_type_clean'] = df_food_model['food_type'].fillna('unknown').astype(str)
food_type_tfidf_vector = TfidfVectorizer(max_features=20, ngram_range=(1, 1))
X_food_type_tfidf  = food_type_tfidf_vector.fit_transform(df_food_model['food_type_clean'])



# Соединение параметров и обучение
X = hstack([X_ingredients_tfidf,X_cooking_method_tfidf,X_food_type_tfidf,X_total_weight])
y = df_food_model['calories']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

"""regressor = PassiveAggressiveRegressor(
    C=0.9,
    max_iter=3000,
    tol=0.001,
    average=True,
    random_state=42
)"""
regressor = XGBRegressor(
    n_estimators=200,
    learning_rate=0.1,
    max_depth=6,
    random_state=42,
    n_jobs=-1
)

regressor.fit(X_train, y_train)
y_pred = regressor.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"MAE: {mae:.1f} ккал")
print(f"R²:  {r2:.3f}")

MAE: 66.8 ккал
R²:  0.823
